In [ ]:
from collections import Counter, defaultdict
from itertools import pairwise
import json
import pandas as pd
import re

In [ ]:
data = pd.read_parquet("all/sentences_parsed.parquet")
data.head()

In [ ]:
data.shape

In [ ]:
targets = pd.read_csv("noun_phrases_sample-classified.csv", sep=';')
targets.head()

In [ ]:
targets.change.unique()

In [ ]:
targets_set = set(targets.noun_phrase.to_list())
targets_set

In [ ]:
len(targets_set)

In [ ]:
count = 0
years_per_target = defaultdict(set)
green_list = []
for i, row in data.iterrows():
    splitted = row.lemmas.split()
    pairs = set([' '.join(pair) for pair in pairwise(splitted)])
    for intersected in pairs.intersection(targets_set):
        green_list.append(
            {"target": intersected, "example": row.sentence, "year": row.year, "doc_id": row.doc_id}
                         )
        count += 1
        years_per_target[intersected].add(row.year)
print(count)
# print(years_per_target)

In [ ]:
targets_single = set()
targets_many = set()
count = 0
for target, year in years_per_target.items():
    if len(year) > 1:
        count += 1
        targets_many.add(target)
    else:
        for word in target.split(): # always 2 words
            targets_single.add(word)
count

In [ ]:
len(targets_single)

In [ ]:
Counter(targets[targets.noun_phrase.isin(targets_many)].change)

Create a file, to which the model will be applied

In [ ]:
seen = set()
lines = []
with open("green.jsonl", "w") as f:
    for line in green_list:
        if (line["target"], line["example"]) not in seen:
            if line["target"] in targets_many:
                f.write(json.dumps(line)+'\n')
                seen.add((line["target"], line["example"]))
                lines.append(line)

Parse model's predictions

In [ ]:
to_split = "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a lexicographer familiar with providing concise definitions of word meanings.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n"

In [ ]:
with open("green.jsonl.cluster-work-projects-nn9851k-mariiaf-periti_2025_data-dpo_model-lora-True-lr-1e-06-lora-16-bf16-linear-beta-0.5-warmup-0.txt", "r") as ours:
    ours_pred = ours.read()
ours_instances = ours_pred.split(to_split)
print(len(ours_instances))

In [ ]:
target_pat = re.compile(f"Please provide a concise definition for the meaning of the word \"(.+)\" in the following sentence:")
to_split_prompt = "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"

In [ ]:
results_our = {"target": [], "prediction": [], "example": [], "year": [], "document": []}

for i, instance_our in enumerate(ours_instances[1:]):
    prompt_our, pred_our = instance_our.split(to_split_prompt)
    
    target_our = re.match(target_pat, prompt_our).group(1)
    example = prompt_our.split(": ", maxsplit=1)[1]
    sentence = data[data.sentence==example]
    year = sentence.year.values[0]
    document = sentence.doc_id.values[0]
    results_our["target"].append(target_our)
    results_our["prediction"].append(pred_our)
    results_our["example"].append(example)
    results_our["year"].append(year)
    results_our["document"].append(document)
results = pd.DataFrame(results_our)
results.head()

In [ ]:
def read_results(path):
    with open(path, "r") as ours:
        ours_pred = ours.read()
        ours_instances = ours_pred.split(to_split)
        print(len(ours_instances))
        results_our = {"target": [], "prediction": [], "example": [], "year": [], "document": []}
        for i, instance_our in enumerate(ours_instances[1:]):
            prompt_our, pred_our = instance_our.split(to_split_prompt)

            target_our = re.match(target_pat, prompt_our).group(1)
            example = prompt_our.split(": ", maxsplit=1)[1]
            sentence = data[data.sentence==example]
            year = sentence.year.values[0]
            document = sentence.doc_id.values[0]
            results_our["target"].append(target_our)
            results_our["prediction"].append(pred_our)
            results_our["example"].append(example)
            results_our["year"].append(year)
            results_our["document"].append(document)
        results = pd.DataFrame(results_our)
        return results

In [ ]:
count_pos = targets[targets.change=="1"].shape[0]
count_maybe_pos = count_pos + targets[targets.change=="x"].shape[0]

In [ ]:
results.to_csv("green.tsv", sep='\t', index=False)

In [ ]:
def study_change(results):
    acc = 0
    acc_strict = 0
    n_targets = 0
    not_changed = []
    table_1 = defaultdict(list)
    table_2 = defaultdict(list)
    for target, t_result in results.groupby("target"):
        print(target)
        year_predictions = []
        target_gold = targets[targets.noun_phrase==target].change.values[0]
        print(target_gold)
        has_changed = False
        n_targets += 1
        defs_dict = defaultdict(list)
        for i, (year, y_result) in enumerate(t_result.groupby("year")):
            print(year)
            defs = set([x.strip() for x in y_result.prediction.unique()])
            for _, sample in y_result.iterrows():
                pred = sample.prediction.strip()
                sense_id = defs_dict.get(pred)
                if sense_id is not None:
                    sense_id = sense_id[0][0]
                else:
                    sense_id = f'{target.replace(" ", "_")}_{len(defs_dict)+1}'
                defs_dict[pred].append((sense_id, sample.example, sample.document, year))
            print(len(defs))
            if i != 0:
                for pred in year_predictions:
                    if pred[1] != defs: # previous year != current
                        print(f"Change between {pred[0]}, {year}")
                        print(f"{pred[0]}: {pred[1]}")
                        print(f"{year}: {defs}")
                        has_changed = True
                        table_2["target"].append(target)
                        table_2["gold"].append(target_gold)
                        table_2["years"].append(f"{pred[0]}, {year}")
                        sense_ids_year_1, sense_ids_year_2, senses_year_1, senses_year_2 = [], [], [], []
                        for previous in sorted(list(pred[1])):
                            sense_ids_year_1.extend([x[0] for x in defs_dict[previous]])
                            senses_year_1.append(previous)
                        for current in sorted(list(defs)):
                            sense_ids_year_2.extend([x[0] for x in defs_dict[current]])
                            senses_year_2.append(current)
                        table_2["sense_ids_year_1"].append(",".join(sorted(list(set(sense_ids_year_1)))))
                        table_2["sense_year_1"].append(",".join(senses_year_1))
                        table_2["sense_ids_year_2"].append(",".join(sorted(list(set(sense_ids_year_2)))))
                        table_2["sense_year_2"].append(",".join(senses_year_2))
            year_predictions.append((year, defs))
            print("---------------------------")
        for sense, sense_data in defs_dict.items():
            print(sense_data)
            table_1["target"].append(target)
            table_1["sense_id"].append(sense_data[0][0])
            table_1["sense"].append(sense)
            example = []
            for instance in sense_data:
                example.append(str(instance[1:]))
            table_1["examples"].append(", ".join(example))
        if has_changed and (target_gold != '0'):
            acc += 1
            if has_changed and (target_gold == '1'):
                acc_strict += 1
        elif not has_changed:
            not_changed.append(target)
    print(acc/count_maybe_pos)
    print(acc_strict/count_pos)
    print(not_changed)
    return pd.DataFrame(table_1), pd.DataFrame(table_2)

In [ ]:
t1, t2 = study_change(results)

In [ ]:
pd.DataFrame(t1)

In [ ]:
pd.DataFrame(t1).to_csv("t1_ours.tsv", sep="\t", index=False)

In [ ]:
pd.DataFrame(t2)

In [ ]:
pd.DataFrame(t2).to_csv("t2_ours.tsv", sep="\t", index=False)

In [ ]:
periti_results = read_results("../green.jsonl.cluster-work-projects-nn9851k-mariiaf-periti_2025_data-LlamaDictionary-en_ML38BI-merged.txt")
periti_results.to_csv("green-periti.tsv", sep='\t', index=False)

In [ ]:
print(periti_results.head())
t1, t2 = study_change(periti_results)

In [ ]:
pd.DataFrame(t1).to_csv("t1_periti.tsv", sep="\t", index=False)
pd.DataFrame(t2).to_csv("t2_periti.tsv", sep="\t", index=False)

In [ ]:
clustered = pd.read_csv("green.clustered.tsv", sep="\t")
clustered.head()

In [ ]:
def study_change(results):
    acc = 0
    acc_strict = 0
    n_targets = 0
    not_changed = []
    table_1 = defaultdict(list)
    table_2 = defaultdict(list)
    for target, t_result in results.groupby("target"):
        print(target)
        year_predictions = []
        target_gold = targets[targets.noun_phrase==target].change.values[0]
        print(target_gold)
        has_changed = False
        n_targets += 1
        defs_dict = defaultdict(list)
        for i, (year, y_result) in enumerate(t_result.groupby("year")):
            print(year)
            defs = set(y_result.labels.unique())
            for _, sample in y_result.iterrows():
                pred = sample.labels
                sense_id = defs_dict.get(pred)
                if sense_id is not None:
                    sense_id = sense_id[0][0]
                else:
                    sense_id = f'{target.replace(" ", "_")}_{len(defs_dict)+1}'
                defs_dict[pred].append((sense_id, sample.example, sample.document, year, sample.prediction))
            print(len(defs))
            if i != 0:
                for pred in year_predictions:
                    if pred[1] != defs: # previous year != current
                        print(f"Change between {pred[0]}, {year}")
                        print(f"{pred[0]}: {pred[1]}")
                        print(f"{year}: {defs}")
                        has_changed = True
                        table_2["target"].append(target)
                        table_2["gold"].append(target_gold)
                        table_2["years"].append(f"{pred[0]}, {year}")
                        sense_ids_year_1, sense_ids_year_2, senses_year_1, senses_year_2 = [], [], [], []
                        for previous in sorted(list(pred[1])):
                            sense_ids_year_1.extend([x[0] for x in defs_dict[previous]])
                            #senses_year_1.append(previous)
                        for current in sorted(list(defs)):
                            sense_ids_year_2.extend([x[0] for x in defs_dict[current]])
                            #senses_year_2.append(current)
                        table_2["sense_ids_year_1"].append(",".join(sorted(list(set(sense_ids_year_1)))))
                        #table_2["sense_year_1"].append(",".join(senses_year_1))
                        table_2["sense_ids_year_2"].append(",".join(sorted(list(set(sense_ids_year_2)))))
                        #table_2["sense_year_2"].append(",".join(senses_year_2))
                        if len(pred[1]) < len(defs):
                            table_2["change_type"].append("gain")
                        elif len(pred[1]) > len(defs):
                            table_2["change_type"].append("loss")
                        else:
                            table_2["change_type"].append("loss and gain")
                    else:
                        has_changed = False
                        table_2["target"].append(target)
                        table_2["gold"].append(target_gold)
                        table_2["years"].append(f"{pred[0]}, {year}")
                        sense_ids_year_1, sense_ids_year_2, senses_year_1, senses_year_2 = [], [], [], []
                        for previous in sorted(list(pred[1])):
                            sense_ids_year_1.extend([x[0] for x in defs_dict[previous]])
                            senses_year_1.append(previous)
                        for current in sorted(list(defs)):
                            sense_ids_year_2.extend([x[0] for x in defs_dict[current]])
                            senses_year_2.append(current)
                        table_2["sense_ids_year_1"].append(",".join(sorted(list(set(sense_ids_year_1)))))
                        #table_2["sense_year_1"].append(",".join(senses_year_1))
                        table_2["sense_ids_year_2"].append(",".join(sorted(list(set(sense_ids_year_2)))))
                        #table_2["sense_year_2"].append(",".join(senses_year_2))
                        table_2["change_type"].append("no change")
            year_predictions.append((year, defs))
            print("---------------------------")
        for sense, sense_data in defs_dict.items():
            print(sense_data)
            table_1["target"].append(target)
            table_1["sense_id"].append(sense_data[0][0])
            table_1["label"].append(sense)
            table_1["sense"].append(", ".join([x[-1].strip() for x in sense_data]))
            example = []
            for instance in sense_data:
                example.append(str(instance[1:-1]))
            table_1["examples"].append(", ".join(example))
        if has_changed and (target_gold != '0'):
            acc += 1
            if has_changed and (target_gold == '1'):
                acc_strict += 1
        elif not has_changed:
            not_changed.append(target)
    print(acc/count_maybe_pos)
    print(acc_strict/count_pos)
    print(not_changed)
    return pd.DataFrame(table_1), pd.DataFrame(table_2)

In [ ]:
t1, t2 = study_change(clustered)

In [ ]:
pd.DataFrame(t1).to_csv("t1_clustered.tsv", sep="\t", index=False)
pd.DataFrame(t2).to_csv("t2_clustered.tsv", sep="\t", index=False)

In [ ]:
clustered = pd.read_csv("green-periti.clustered.tsv", sep="\t")
clustered.head()

In [ ]:
t1, t2 = study_change(clustered)

In [ ]:
pd.DataFrame(t1).to_csv("t1_clustered-periti.tsv", sep="\t", index=False)
pd.DataFrame(t2).to_csv("t2_clustered-periti.tsv", sep="\t", index=False)